# S44 답이 없는 부분 밝히기: v19 결과

기존 줄에 구절을 더한 프롬프트 v19를 기준 v16과 비교해, 기준 반영을 정할 근거를 낸다(`docs/slices/44-answer-gaps.md`, D-83). 저장된 기록만 읽고 API를 부르지 않는다. 판정 세트는 지표와 코드 집계만 쓰고, 답변·질의 원문은 출력하지 않는다.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.gen_eval.checks import CITE, sentences

In [2]:
ROOT = Path.cwd().parent
STEM = "hybrid-cond-filter-law-ms256-k20-ce-{v}-dev-full-spk-marks{s}"
RUNS = ROOT / "data/gen_eval/runs"
JUDGE = ROOT / "data/gen_eval/judge"
TAIL = "-deepseek_go_mixed-j_v13.jsonl"
COMPARE = ROOT / "data/gen_eval/compare/s44-v19-vs-exp026.json"
READS = ROOT / "data/gen_eval/error_analysis/s44_compare"
KEY = "찾지 못"
LIMITS = {"composite": ("상한0", 0.0), "faithfulness": ("하한", -0.05), "relevance": ("하한", -0.035), "over_refusal": ("상한", 0.02), "speaker_mismatch": ("상한", 0.02)}

In [3]:
def load(path):
    with path.open(encoding="utf-8") as f:
        return {r["qid"]: r for r in map(json.loads, f)}


def run(version, suffix=""):
    return load(RUNS / (STEM.format(v=version, s=suffix) + ".jsonl"))


def judged(version, suffix=""):
    return load(JUDGE / (STEM.format(v=version, s=suffix) + TAIL))


def passes(name, low, high):
    kind, limit = LIMITS[name]
    if kind == "하한":
        return low >= limit
    return high >= limit if kind == "상한0" else high <= limit


def citation_row(name, records):
    no_cite = uncited = total = 0
    for r in records.values():
        body = [s for s in sentences(r["answer"]) if KEY not in s]
        no_cite += bool(body) and not CITE.search(r["answer"])
        uncited += sum(not CITE.search(s) for s in body)
        total += len(body)
    return {"설정": name, "인용 전무": no_cite, "인용 없는 문장": uncited, "문장": total, "비율": round(uncited / total, 3), "안내 답변": sum(KEY in r["answer"] for r in records.values()), "글자 평균": sum(len(r["answer"]) for r in records.values()) // len(records)}

## 판정 지표

`rag.gen_eval.compare` 결과(기준 v16). 통과 칸은 지시서 결정 사항 1이다.

In [4]:
result = json.loads(COMPARE.read_text(encoding="utf-8"))
rows = []
for name, m in result["metrics"].items():
    low, high = m["ci"]
    rows.append({"지표": name, "기준": round(m["base"], 3), "후보": round(m["cand"], 3), "차이": round(m["diff"], 4), "하한": round(low, 4), "상한": round(high, 4), "질의": m["n"], "통과": passes(name, low, high) if name in LIMITS else ""})
print("compare", result["verdict"], result["failed"], "judge 설정 차이", result["check_settings_diff"])
print("충실성 형식 오류 후보", result["exclusions"]["cand"]["faithfulness"]["format_errors"], "기준", result["exclusions"]["base"]["faithfulness"]["format_errors"])
pd.DataFrame(rows)

compare 보류 ['종합 지표 구간 하한 ≤ 0', '종합 지표 점 추정 < 0.02'] judge 설정 차이 {}
충실성 형식 오류 후보 7 기준 4


,지표,기준,후보,차이,하한,상한,질의,통과
0,faithfulness,0.990,0.985,-0.0045,-0.0129,0.0033,319,True
1,relevance,0.877,0.884,0.0076,-0.0109,0.0266,328,True
2,composite,0.933,0.935,0.0017,-0.0087,0.0123,319,True
3,inclusion,0.714,0.735,0.0211,-0.0039,0.0455,266,
4,over_refusal,0.003,0.003,0.0000,-0.0052,0.0051,286,True
5,speaker_mismatch,0.003,0.007,0.0032,-0.0006,0.0079,319,True


## 판정 세트 코드 집계

인용과 안내 문장은 찾지 못했다는 문장을 뺀 값이다. 재생성은 `gen_rep=1`이다.

In [5]:
base, cand, rerun = run("v16"), run("v19"), run("v19", "-r1")
pd.DataFrame([citation_row("v16", base), citation_row("v19", cand), citation_row("v19 재생성", rerun)])

,설정,인용 전무,인용 없는 문장,문장,비율,안내 답변,글자 평균
0,v16,5,19,860,0.022,21,324
1,v19,7,43,877,0.049,31,335
2,v19 재생성,5,25,870,0.029,31,334


In [6]:
jb, jc = judged("v16"), judged("v19")
rows = []
for name, j in (("v16", jb), ("v19", jc)):
    outcomes = Counter(i["outcome"] for r in j.values() for i in r["item_eval"]["items"])
    refusal = Counter(str(r.get("refusal")) for r in j.values())
    rows.append({"설정": name, "전체 거부": refusal["full"], "부분 거부": refusal["partial"], **{k: outcomes[k] for k in ("근거 없이 답함", "안내 없이 부분 답함", "무언 생략", "누락", "과잉 거부", "상황 전달")}})
pd.DataFrame(rows)

,설정,전체 거부,부분 거부,근거 없이 답함,안내 없이 부분 답함,무언 생략,누락,과잉 거부,상황 전달
0,v16,1,20,59,51,27,15,2,42
1,v19,2,29,58,53,14,10,2,42


## 세트 밖 100질의

겨냥한 안내가 얼마나 붙었는지와 서브에이전트의 비교 판독이다. 판독은 달라진 답변 34질의(안내가 새로 붙음, 겨냥 유형, 잘린 발언, 종합이 낮아짐)만 보았고 사람 검수를 거치지 않았다.

In [7]:
oa, ob = run("v16", "-outset100"), run("v19", "-outset100")
pd.DataFrame([citation_row("v16 세트 밖", oa), citation_row("v19 세트 밖", ob)])

,설정,인용 전무,인용 없는 문장,문장,비율,안내 답변,글자 평균
0,v16 세트 밖,0,0,272,0.000,5,339
1,v19 세트 밖,3,11,267,0.041,8,337


In [8]:
reads = [json.loads(p.read_text(encoding="utf-8-sig")) for p in sorted(READS.glob("*.json"))]
table = pd.DataFrame(reads)
print("판독", len(table))
print(table.verdict.value_counts().to_dict())
print(table.metric_cause.value_counts().to_dict())
print({k: int(table[k].astype(str).eq("True").sum()) for k in ("false_absent", "gap_disclosed", "lost_content", "new_error")})
new = {q for q in ob if KEY in ob[q]["answer"] and KEY not in oa[q]["answer"]}
print("안내가 새로 붙은 답변", len(new), "그 판독", table[table.qid.isin(new)].verdict.value_counts().to_dict())

판독 34
{'같음': 19, '나아짐': 10, '나빠짐': 5}
{'해당 없음': 23, 'judge 적용 흔들림': 7, '답변이 나빠짐': 3, '그 밖': 1}
{'false_absent': 1, 'gap_disclosed': 5, 'lost_content': 10, 'new_error': 2}
안내가 새로 붙은 답변 3 그 판독 {'나아짐': 2, '같음': 1}


## 재생성 흔들림과 비교

같은 설정을 다시 생성해도 값이 움직인다. v16과 v19를 각각의 재생성과 함께 놓아, 차이가 지시 때문인지 흔들림 안인지 본다.

In [9]:
base_r = run("v16", "-r1")
pd.DataFrame([citation_row("v16", base), citation_row("v16 재생성", base_r), citation_row("v19", cand), citation_row("v19 재생성", rerun)])

,설정,인용 전무,인용 없는 문장,문장,비율,안내 답변,글자 평균
0,v16,5,19,860,0.022,21,324
1,v16 재생성,8,35,869,0.040,19,331
2,v19,7,43,877,0.049,31,335
3,v19 재생성,5,25,870,0.029,31,334
